In [1]:
import pyspark, os, sys, shutil
from pyspark.sql import SparkSession
os.environ['PYSPARK_PYTHON'] = os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
spark = SparkSession.builder.appName("FileSourcesFormats").getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")
print("Spark version:", spark.version)

# Clean up output directory from previous runs
if os.path.exists("tmp/formats"):
    shutil.rmtree("tmp/formats")
os.makedirs("tmp/formats", exist_ok=True)

# ── Sample dataset: 500 employee records ─────────────────────────────────
NAMES = ["Alice","Bob","Carol","Dave","Eve","Frank","Grace","Hank","Ivy","Jack",
         "Kate","Liam","Maya","Noah","Olivia","Paul","Quinn","Rosa","Sam","Tina"]
DEPTS = ["Engineering","Marketing","HR","Finance","Operations"]

rows = [(i + 1, NAMES[i % 20], DEPTS[i % 5],
         50000 + (i * 1327 % 60000), 25 + (i * 7 % 30))
        for i in range(500)]

df = spark.createDataFrame(rows, ["id", "name", "dept", "salary", "age"])
df.show(5)
print(f"Dataset: {df.count()} rows x {len(df.columns)} columns")
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/28 18:28:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.2


+---+-----+-----------+------+---+
| id| name|       dept|salary|age|
+---+-----+-----------+------+---+
|  1|Alice|Engineering| 50000| 25|
|  2|  Bob|  Marketing| 51327| 32|
|  3|Carol|         HR| 52654| 39|
|  4| Dave|    Finance| 53981| 46|
|  5|  Eve| Operations| 55308| 53|
+---+-----+-----------+------+---+
only showing top 5 rows
Dataset: 500 rows x 5 columns


# File Sources and Formats

Spark SQL can read and write data in many file formats through a unified **DataFrame interface**. The same `spark.read` / `df.write` API works across all formats — only the `.format()` call (or shorthand method) changes.

```python
# Generic form
df = spark.read.format("csv").option("header", "true").load("path/")
df.write.format("parquet").mode("overwrite").save("out/")

# Shorthand
df = spark.read.csv("path/", header=True)
df.write.parquet("out/")
```

This session covers the four most common formats in data engineering pipelines:

| Format | Storage layout | Typical use |
|---|---|---|
| **Text** | Row-based, unstructured | Log files, raw ingest |
| **CSV** | Row-based, delimited | Interoperability, small datasets |
| **JSON** | Row-based, self-describing | APIs, nested/semi-structured data |
| **Parquet** | **Column-based**, compressed | Analytics, data lakes, default Spark format |

**Reading:** Apache Spark 4.1.2 — Data Sources (https://spark.apache.org/docs/latest/sql-data-sources.html)

# Example 1: Generic Load/Save API and Save Modes

All formats share the same read/write pattern. The `format()` call accepts short names: `"text"`, `"csv"`, `"json"`, `"parquet"`, `"orc"`, `"jdbc"` (Apache Spark Documentation, 2024).

**Save Modes** control what happens when the output path already exists:

| Mode | Behaviour |
|---|---|
| `"error"` (default) | Raise exception if data exists |
| `"overwrite"` | Delete existing data and replace |
| `"append"` | Add to existing data |
| `"ignore"` | Silently skip if data exists (like `CREATE IF NOT EXISTS`) |

Save modes are **not atomic** and do not use locking (Apache Spark Documentation, 2024).

In [3]:
# 1a. Two equivalent ways to read a format

# Generic form  (always works)
df_generic = (spark.read
              .format("parquet")
              .load("data/Cambridge_Budget_Salaries.parquet"))

# Shorthand  (specific to each format)
df_short   = spark.read.parquet("data/Cambridge_Budget_Salaries.parquet")

# 1b. Save mode demonstration
df.write.mode("overwrite").parquet("tmp/formats/demo_parquet")
print("First write: OK")

df.write.mode("overwrite").parquet("tmp/formats/demo_parquet")   # no error
print("Second write with overwrite: OK")

df.write.mode("ignore").parquet("tmp/formats/demo_parquet")      # silently skips
print("Third write with ignore: OK (data unchanged)")

# 1c. Run SQL directly on a file without loading it first
result = spark.sql(
    "SELECT * FROM parquet.`tmp/formats/demo_parquet` LIMIT 3"
)
result.show()

First write: OK
Second write with overwrite: OK
Third write with ignore: OK (data unchanged)
+---+-----+-----------+------+---+
| id| name|       dept|salary|age|
+---+-----+-----------+------+---+
|  1|Alice|Engineering| 50000| 25|
|  2|  Bob|  Marketing| 51327| 32|
|  3|Carol|         HR| 52654| 39|
+---+-----+-----------+------+---+



# Example 2: Text Files

Text format is the simplest: Spark reads each line as one row with a single column named **`value`** (type `StringType`). There is no schema inference — every line is returned as-is (Apache Spark Documentation, 2024).

Use cases: raw log ingestion, custom parsing pipelines, line-by-line regex processing.

In [4]:
# Write: each row is formatted as a string and written as one line.
# Note: text format requires exactly ONE string column.

from pyspark.sql.functions import concat_ws, col

# Create a single-column string representation
text_df = df.select(
    concat_ws(",", col("id").cast("string"), col("name"),
              col("dept"), col("salary").cast("string"),
              col("age").cast("string")).alias("value")
)
text_df.write.mode("overwrite").text("tmp/formats/employees_text")

# Read: every line becomes one row in the "value" column
text_read = spark.read.text("tmp/formats/employees_text")
text_read.printSchema()
text_read.show(5, truncate=False)

root
 |-- value: string (nullable = true)

+------------------------------+
|value                         |
+------------------------------+
|451,Kate,Engineering,107150,25|
|452,Liam,Marketing,108477,32  |
|453,Maya,HR,109804,39         |
|454,Noah,Finance,51131,46     |
|455,Olivia,Operations,52458,53|
+------------------------------+
only showing top 5 rows


# Example 3: CSV Files

CSV is the most widely used interchange format. Key options (Apache Spark Documentation, 2024):

| Option | Default | Meaning |
|---|---|---|
| `header` | `false` | First row is a header |
| `inferSchema` | `false` | Auto-detect column types (requires extra scan) |
| `sep` | `,` | Field delimiter |
| `nullValue` | `""` | String to interpret as null |
| `timestampFormat` | ISO 8601 | Format for timestamp columns |

**Limitation**: Spark reads entire rows to access any column — there is no way to skip columns you do not need.

In [5]:
# Write CSV with a header row
df.write.mode("overwrite") \
    .option("header", "true") \
    .csv("tmp/formats/employees_csv")

# Read back — specify header and inferSchema so types are preserved
csv_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("tmp/formats/employees_csv")

csv_df.printSchema()
csv_df.show(5)

# Custom delimiter (semicolon) — useful for European locales
df.write.mode("overwrite") \
    .option("header", "true") \
    .option("sep", ";") \
    .csv("tmp/formats/employees_csv_semi")

spark.read.option("header","true").option("sep",";").option("inferSchema","true") \
    .csv("tmp/formats/employees_csv_semi").show(3)

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- age: integer (nullable = true)

+---+-----+-----------+------+---+
| id| name|       dept|salary|age|
+---+-----+-----------+------+---+
|401|Alice|Engineering|100800| 35|
|402|  Bob|  Marketing|102127| 42|
|403|Carol|         HR|103454| 49|
|404| Dave|    Finance|104781| 26|
|405|  Eve| Operations|106108| 33|
+---+-----+-----------+------+---+
only showing top 5 rows
+---+----+-----------+------+---+
| id|name|       dept|salary|age|
+---+----+-----------+------+---+
|451|Kate|Engineering|107150| 25|
|452|Liam|  Marketing|108477| 32|
|453|Maya|         HR|109804| 39|
+---+----+-----------+------+---+
only showing top 3 rows


# Example 4: JSON Files

Spark expects **one JSON object per line** (JSONL / newline-delimited JSON) by default. Schema is inferred by sampling the data. Use `multiLine=True` for pretty-printed files (Apache Spark Documentation, 2024).

**Strength**: JSON handles nested structures (objects within objects, arrays) naturally — something CSV cannot express. Spark will automatically expand nested fields into a struct column.

**Weakness**: Every row repeats all field names as keys, making JSON the most verbose (largest file size) of the structured formats.

In [6]:
# Write JSON — schema is embedded in every row
df.write.mode("overwrite").json("tmp/formats/employees_json")

# Read back — schema is inferred from the data
json_df = spark.read.json("tmp/formats/employees_json")
json_df.printSchema()
json_df.show(5)

# JSON excels at nested/semi-structured data
nested_json = [
    {"order_id": 1, "customer": {"name": "Alice", "city": "Atlanta"},
     "items": [{"sku": "A1", "qty": 2}, {"sku": "B3", "qty": 1}], "total": 145.0},
    {"order_id": 2, "customer": {"name": "Bob",   "city": "Boston"},
     "items": [{"sku": "C2", "qty": 5}],                             "total": 250.0},
]
import json, os
os.makedirs("tmp/formats/orders_json", exist_ok=True)
with open("tmp/formats/orders_json/orders.json", "w") as f:
    for record in nested_json:
        f.write(json.dumps(record) + "\n")

orders_df = spark.read.json("tmp/formats/orders_json")
orders_df.printSchema()   # customer and items are structs/arrays
orders_df.select("order_id", "customer.name", "customer.city", "total").show()

root
 |-- age: long (nullable = true)
 |-- dept: string (nullable = true)
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: long (nullable = true)

+---+-----------+---+------+------+
|age|       dept| id|  name|salary|
+---+-----------+---+------+------+
| 25|Engineering|451|  Kate|107150|
| 32|  Marketing|452|  Liam|108477|
| 39|         HR|453|  Maya|109804|
| 46|    Finance|454|  Noah| 51131|
| 53| Operations|455|Olivia| 52458|
+---+-----------+---+------+------+
only showing top 5 rows
root
 |-- customer: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- name: string (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- qty: long (nullable = true)
 |    |    |-- sku: string (nullable = true)
 |-- order_id: long (nullable = true)
 |-- total: double (nullable = true)

+--------+-----+-------+-----+
|order_id| name|   city|total|
+--------+-----+-------+-----+
|    

# Example 5: Row-Based vs Column-Based Storage

## The Core Trade-off

**Row-based** formats (text, CSV, JSON) store one complete record contiguously:

```
Disk layout:
[id1, name1, dept1, salary1, age1]  ← row 1 (all columns together)
[id2, name2, dept2, salary2, age2]  ← row 2
[id3, name3, dept3, salary3, age3]  ← row 3

Query: SELECT AVG(salary) FROM employees
→ Must read ALL bytes of ALL rows — including id, name, dept, age
→ 100% of data scanned for a 1-column query
```

**Column-based** formats (Parquet, ORC) store each column contiguously:

```
Disk layout:
[id1,     id2,     id3    ]  ← all id values
[name1,   name2,   name3  ]  ← all name values
[dept1,   dept2,   dept3  ]  ← all dept values
[salary1, salary2, salary3]  ← all salary values  ← ONLY THIS IS READ
[age1,    age2,    age3   ]  ← all age values

Query: SELECT AVG(salary) FROM employees
→ Spark reads ONLY the salary column — 80% less I/O!
```

## Why Columnar Formats Compress Better

Values in the same column tend to be similar (e.g., `dept` is one of five strings repeated 100 times). Run-length encoding and dictionary compression work far more effectively on columnar data than on interleaved row data.

## When to Use Each

| Scenario | Recommended format |
|---|---|
| Receiving data from external system | CSV, JSON (interoperability) |
| Storing data in your data lake | **Parquet** |
| Querying a few columns of a wide table | **Parquet** (column pruning) |
| Streaming raw logs | Text |
| Nested/hierarchical data | JSON |
| Cross-platform analytics | **Parquet** (de facto standard) |

In [7]:
# Empirical file size comparison: same 500-row dataset in four formats

def dir_size_bytes(path):
    """Sum sizes of all data files in a directory (ignore _SUCCESS, .crc, etc.)."""
    total = 0
    for fname in os.listdir(path):
        if not fname.startswith("_") and not fname.startswith("."):
            total += os.path.getsize(os.path.join(path, fname))
    return total

sizes = {}
for fmt, path in [
    ("Text",    "tmp/formats/employees_text"),
    ("CSV",     "tmp/formats/employees_csv"),
    ("JSON",    "tmp/formats/employees_json"),
    ("Parquet", "tmp/formats/demo_parquet"),
]:
    sizes[fmt] = dir_size_bytes(path)

baseline = sizes["CSV"]
print("{:>10}  {:>14}  {:>8}".format("Format", "Size (bytes)", "vs CSV"))
print("-" * 40)
for fmt, sz in sizes.items():
    ratio = sz / baseline
    bar   = "█" * int(ratio * 20)
    print(f"{fmt:>10}  {sz:>14,}  {ratio:>6.1f}x  {bar}")
print()
print("Parquet is typically 3-10x smaller than CSV for real-world data.")
print("JSON is often the largest due to repeated field name keys in each row.")

    Format    Size (bytes)    vs CSV
----------------------------------------
      Text          13,451     1.0x  ███████████████████
       CSV          13,691     1.0x  ████████████████████
      JSON          33,451     2.4x  ████████████████████████████████████████████████
   Parquet          24,582     1.8x  ███████████████████████████████████

Parquet is typically 3-10x smaller than CSV for real-world data.
JSON is often the largest due to repeated field name keys in each row.


# Example 6: Parquet — Column Pruning and Predicate Pushdown

Parquet is the **default format** for Spark SQL and the de facto standard for data lakes (Apache Spark Documentation, 2024).

Key advantages:
- **Self-describing**: schema is stored in the file footer — no separate schema definition needed.
- **Column pruning**: when you `SELECT` only some columns, Spark reads only those columns from disk.
- **Predicate pushdown**: `WHERE` clauses are pushed into the Parquet reader, skipping row groups that cannot match.
- **Compression**: Snappy by default; also supports gzip, zstd.
- **Schema evolution**: supports adding nullable columns via schema merging.

In [8]:
# 6a. Write and read Parquet — schema is preserved automatically
df.write.mode("overwrite").parquet("tmp/formats/employees_parquet")

parquet_df = spark.read.parquet("tmp/formats/employees_parquet")
parquet_df.printSchema()   # types are exact — no need for inferSchema
parquet_df.show(5)

# 6b. Column pruning — Spark reads only "salary" from disk
#     The physical plan shows "ReadSchema: struct<salary:bigint>"
print("=== Column-pruned read (only salary) ===")
parquet_df.select("salary").explain()

# 6c. Predicate pushdown — filter is pushed into the Parquet reader
#     The plan shows "PushedFilters: [IsNotNull(dept), EqualTo(dept,Engineering)]"
print("=== Filter pushdown (dept = Engineering) ===")
parquet_df.filter("dept = 'Engineering'").select("name","salary").explain()

# 6d. Compression codec — default is snappy; change at write time
df.write.mode("overwrite") \
    .option("compression", "gzip") \
    .parquet("tmp/formats/employees_parquet_gzip")
print("Written with gzip compression.")

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- age: long (nullable = true)

+---+-----+-----------+------+---+
| id| name|       dept|salary|age|
+---+-----+-----------+------+---+
|  1|Alice|Engineering| 50000| 25|
|  2|  Bob|  Marketing| 51327| 32|
|  3|Carol|         HR| 52654| 39|
|  4| Dave|    Finance| 53981| 46|
|  5|  Eve| Operations| 55308| 53|
+---+-----+-----------+------+---+
only showing top 5 rows
=== Column-pruned read (only salary) ===
== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [salary#227L] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/app/work/data-engineering-preparation/tmp/formats/employees_parq..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<salary:bigint>


=== Filter pushdown (dept = Engineering) ===
== Physical Plan ==
*(1) Project [name#225, salary#227L]
+- *(1) Filter (isnotnull(de

# Example 7: Partition Discovery

`partitionBy()` writes data into subdirectories named `column=value`. Spark automatically infers the partition columns when reading, adding them to the schema (Apache Spark Documentation, 2024).

When you filter on a partition column, Spark reads **only the relevant subdirectory** — this is called **partition pruning** and eliminates I/O entirely for the other partitions.

```
employees_by_dept/
  dept=Engineering/  ← read only this when filtering dept = "Engineering"
    part-00000.parquet
  dept=Finance/
  dept=HR/
  dept=Marketing/
  dept=Operations/
```

In [9]:
# 7a. Write partitioned by dept
df.write.mode("overwrite") \
    .partitionBy("dept") \
    .parquet("tmp/formats/employees_by_dept")

# 7b. Inspect the directory structure
print("Partition directories:")
for entry in sorted(os.listdir("tmp/formats/employees_by_dept")):
    print(f"  {entry}/")

# 7c. Read back — dept is automatically added as a column
partitioned_df = spark.read.parquet("tmp/formats/employees_by_dept")
partitioned_df.printSchema()   # dept appears as a column from the path

# 7d. Filter on partition key — Spark reads ONLY the Engineering directory
eng_df = partitioned_df.filter("dept = 'Engineering'")
print(f"Engineering employees: {eng_df.count()}")
eng_df.show(5)

# Show plan: PartitionFilters in the Parquet scan — no other dirs are touched
eng_df.explain()

Partition directories:
  ._SUCCESS.crc/
  _SUCCESS/
  dept=Engineering/
  dept=Finance/
  dept=HR/
  dept=Marketing/
  dept=Operations/
root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- age: long (nullable = true)
 |-- dept: string (nullable = true)

Engineering employees: 100
+---+-----+------+---+-----------+
| id| name|salary|age|       dept|
+---+-----+------+---+-----------+
|301|Alice| 88100| 25|Engineering|
|306|Frank| 94735| 30|Engineering|
|311| Kate|101370| 35|Engineering|
|316| Paul|108005| 40|Engineering|
|321|Alice| 54640| 45|Engineering|
+---+-----+------+---+-----------+
only showing top 5 rows
== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [id#247L,name#248,salary#249L,age#250L,dept#251] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/app/work/data-engineering-preparation/tmp/formats/employees_by_d..., PartitionFilters: [isnotnull(dept#251), (dept#251 

# Example 8: Format Comparison Summary

| | Text | CSV | JSON | Parquet |
|---|---|---|---|---|
| **Storage layout** | Row | Row | Row | **Column** |
| **Human-readable** | ✓ | ✓ | ✓ | ✗ (binary) |
| **Schema in file** | ✗ | Optional header | ✓ (keys) | **✓ (footer)** |
| **Nested data** | ✗ | ✗ | ✓ | ✓ (structs/arrays) |
| **Compression** | None | Optional | Optional | **Built-in (Snappy)** |
| **Column pruning** | ✗ | ✗ | ✗ | **✓** |
| **Predicate pushdown** | ✗ | ✗ | ✗ | **✓** |
| **Schema evolution** | ✗ | Manual | Flexible | **✓** |
| **Typical file size** | Large | Medium | Largest | **Smallest** |
| **Default for Spark** | No | No | No | **Yes** |

**Rule of thumb**: receive data as CSV/JSON, store and query as Parquet.

In [10]:
# Final size comparison including gzip Parquet
import os

formats = [
    ("Text",            "tmp/formats/employees_text"),
    ("CSV",             "tmp/formats/employees_csv"),
    ("JSON",            "tmp/formats/employees_json"),
    ("Parquet (snappy)","tmp/formats/employees_parquet"),
    ("Parquet (gzip)",  "tmp/formats/employees_parquet_gzip"),
]

print("{:<20}  {:>10}  Relative to CSV".format("Format", "Bytes"))
print("-" * 52)

csv_sz = dir_size_bytes("tmp/formats/employees_csv")
for label, path in formats:
    sz = dir_size_bytes(path)
    pct = sz / csv_sz * 100
    bar = "█" * max(1, int(pct / 5))
    print(f"{label:<20}  {sz:>10,}  {pct:5.0f}%  {bar}")

Format                     Bytes  Relative to CSV
----------------------------------------------------
Text                      13,451     98%  ███████████████████
CSV                       13,691    100%  ████████████████████
JSON                      33,451    244%  ████████████████████████████████████████████████
Parquet (snappy)          24,582    180%  ███████████████████████████████████
Parquet (gzip)            22,689    166%  █████████████████████████████████


In [11]:
spark.stop()